# Stage 7: ROAD Baseline Models

Trains the same Random Forest and 1D-CNN baselines used for CICIoV2024 on the
ROAD dataset, loading the processed arrays saved by notebook 06.

Both models are evaluated on the held-out signature-level test set. The clean
baseline establishes detectability before adversarial perturbation is introduced.

In [1]:
# This notebook lives in notebooks/; code lives in src/
import sys
from pathlib import Path

SRC = Path.cwd().parent / "src"
sys.path.insert(0, str(SRC))

import config
import models
import evaluation

import numpy as np
import joblib
import torch

# --- Device: use the GPU if available, else fall back to CPU ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device      :", DEVICE)
if DEVICE == "cuda":
    print("GPU         :", torch.cuda.get_device_name(0))
print("PyTorch     :", torch.__version__)

Device      : cuda
GPU         : NVIDIA GeForce RTX 4080 Laptop GPU
PyTorch     : 2.5.1+cu121


In [2]:
# Load the processed ROAD arrays saved by notebook 06
arrays = np.load(config.PROCESSED_DIR / "road_stage2_arrays.npz")
X_train = arrays["X_train"]
y_train = arrays["y_train"]
X_test  = arrays["X_test"]
y_test  = arrays["y_test"]

# Load the fitted label encoder for class names in report order
road_encoder = joblib.load(config.PROCESSED_DIR / "road_label_encoder.joblib")
class_names = list(road_encoder.classes_)

print("X_train:", X_train.shape, " X_test:", X_test.shape)
print("classes:", class_names)

X_train: (31886, 9)  X_test: (7972, 9)
classes: ['benign', 'fuzzing', 'max-speedometer', 'reverse-light-off', 'reverse-light-on']


## Random Forest baseline

scikit-learn Random Forest (CPU; RF is not GPU-accelerated). Same configuration
as the CICIoV2024 baseline for a like-for-like comparison.

In [3]:
# Random Forest (CPU: scikit-learn RF is not GPU-accelerated)
rf = models.build_random_forest(random_seed=config.RANDOM_SEED)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

rf_metrics = evaluation.evaluate_model(
    y_test, rf_pred, class_names, model_name="ROAD Random Forest",
)


    ROAD Random Forest
    Accuracy    : 1.0000
    Macro-F1    : 1.0000

   Per-class report:
                   precision    recall  f1-score   support

           benign       1.00      1.00      1.00      4238
          fuzzing       1.00      1.00      1.00       118
  max-speedometer       1.00      1.00      1.00      2112
reverse-light-off       1.00      1.00      1.00       305
 reverse-light-on       1.00      1.00      1.00      1199

         accuracy                           1.00      7972
        macro avg       1.00      1.00      1.00      7972
     weighted avg       1.00      1.00      1.00      7972

     Confusion matrix (rows=true, cols=pred):
[[4238    0    0    0    0]
 [   0  118    0    0    0]
 [   0    0 2112    0    0]
 [   0    0    0  305    0]
 [   0    0    0    0 1199]]


## 1D-CNN baseline

Same CNN1D architecture as CICIoV2024, trained on the GPU. Feature count is 9
(ID + 8 payload bytes), so the network geometry is unchanged from CICIoV.

In [4]:
# 1D-CNN, trained on the GPU
cnn = models.CNN1D(n_features=X_train.shape[1], n_classes=len(class_names))
cnn = models.train_cnn(
    cnn, X_train, y_train,
    n_epochs=50, device=DEVICE, random_seed=config.RANDOM_SEED,
)

# Evaluate on the test set (move test tensor to the same device)
cnn.eval()
with torch.no_grad():
    X_test_t = torch.tensor(X_test, dtype=torch.float32, device=DEVICE)
    cnn_pred = cnn(X_test_t).argmax(dim=1).cpu().numpy()

cnn_metrics = evaluation.evaluate_model(
    y_test, cnn_pred, class_names, model_name="ROAD 1D-CNN",
)

    epoch   1/50     loss 0.3212
    epoch   5/50     loss 0.0357
    epoch  10/50     loss 0.0171
    epoch  15/50     loss 0.0147
    epoch  20/50     loss 0.0111
    epoch  25/50     loss 0.0095
    epoch  30/50     loss 0.0118
    epoch  35/50     loss 0.0079
    epoch  40/50     loss 0.0059
    epoch  45/50     loss 0.0070
    epoch  50/50     loss 0.0061

    ROAD 1D-CNN
    Accuracy    : 0.9990
    Macro-F1    : 0.9964

   Per-class report:
                   precision    recall  f1-score   support

           benign       1.00      1.00      1.00      4238
          fuzzing       0.98      1.00      0.99       118
  max-speedometer       1.00      1.00      1.00      2112
reverse-light-off       0.99      0.99      0.99       305
 reverse-light-on       1.00      1.00      1.00      1199

         accuracy                           1.00      7972
        macro avg       1.00      1.00      1.00      7972
     weighted avg       1.00      1.00      1.00      7972

     Confusion

## Save baseline metrics

Persists both models' metrics to results/ for the write-up and for comparison
against the adversarial results in the next stage.

In [5]:
import json

baseline_report = {
    "dataset": "ROAD",
    "device": DEVICE,
    "n_classes": len(class_names),
    "class_names": class_names,
    "test_size": int(len(y_test)),
    "random_forest": {
        "accuracy": rf_metrics["accuracy"],
        "macro_f1": rf_metrics["macro_f1"],
        "confusion_matrix": rf_metrics["confusion_matrix"],
    },
    "cnn_1d": {
        "accuracy": cnn_metrics["accuracy"],
        "macro_f1": cnn_metrics["macro_f1"],
        "confusion_matrix": cnn_metrics["confusion_matrix"],
    },
}

config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
path = config.RESULTS_DIR / "road_baseline_metrics.json"
with open(path, "w") as f:
    json.dump(baseline_report, f, indent=2)
print("saved baseline metrics ->", path)

saved baseline metrics -> /home/koala/lab/adversec/results/road_baseline_metrics.json
